[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/29_adam.ipynb)

# 🟠 中等：Adam 优化器

从零实现 **Adam** 优化器。

### 函数签名
```python
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8): ...
    def step(self): ...
    def zero_grad(self): ...
```

### 算法 (per parameter)
```
m = β1 * m + (1-β1) * grad
v = β2 * v + (1-β2) * grad²
m̂ = m / (1 - β1ᵗ)    # bias correction
v̂ = v / (1 - β2ᵗ)
p -= lr * m̂ / (√v̂ + ε)
```

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ 在此实现你的代码

class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        pass  # 存储 params，初始化 m 和 v 为零

    def step(self):
        pass  # 使用 Adam 规则更新 params

    def zero_grad(self):
        pass  # 清零所有梯度

<strong>随机梯度下降(Stochastic Gradient Descent, SGD)</strong>的核心思想与 BGD 恰好相反：**每次参数更新时，不再计算整个数据集的梯度，而是仅随机抽取一个样本 \( (x^{(i)}, y^{(i)}) \) 来计算梯度**。

---

### 1. 参数更新公式（通用形式）

$\theta = \theta - \eta \cdot abla J_i(\theta)$

- **$\theta$**：模型参数。
- **$\eta$**（学习率）：控制步长。
- **$abla J_i(\theta)$**：**仅基于第$i$个样本**计算出的损失函数梯度。

---

### 2. 在线性回归中的具体公式

假设线性回归的损失函数为均方误差（针对**单个样本** $i$）
$J_i(\theta) = \frac{1}{2} \left( h_\theta(x^{(i)}) - y^{(i)} \right)^2$
那么，SGD 对第 \( j \) 个参数的**迭代更新公式**为：
$\theta_j = \theta_j - \eta \cdot \left( h_\theta(x^{(i)}) - y^{(i)} \right) \cdot x_j^{(i)}$

**关键点解读：**

- **没有求和符号 $\sum $**：公式中只出现第 $i$ 个样本的特征 $x_j^{(i)}$ 和标签 $y^{(i)}$。
- **更新频率极高**：每处理 **1 个**样本，参数就更新一次。遍历完整个数据集（1个 Epoch）时，参数已经更新了 m 次（ $m$ 为样本总数）。

---

### 3. 与 BGD 的本质区别（核心对比）

| 对比维度 | **SGD（随机梯度下降）** | **BGD（批量梯度下降）** |
| :--- | :--- | :--- |
| **每次更新所用样本** | 1 个样本 | 全部 $m $个样本 |
| **更新公式中的运算** | 无求和，单点计算 | 包含 $\sum_{i=1}^{m}$ 全量求和 |
| **计算速度** | 极快（单次迭代） | 极慢（大数据集下） |
| **收敛路径** | 震荡剧烈，呈“锯齿状” | 平滑稳定，笔直走向最优点 |
| **收敛效果** | 无法收敛到精确最小值，会在最优点附近波动 | 能稳定收敛到全局最优点（凸函数） |
| **跳出局部最优** | **更容易**（震荡特性有助于逃离局部极小点和鞍点） | 容易陷入局部最优 |

---

### 4. 一个重要补充：实际工程中的“SGD”

在现代深度学习框架（如 PyTorch、TensorFlow）中，当你调用 `torch.optim.SGD` 时，**它通常并不是严格的“1个样本更新一次”**。

实际使用时，你传入的往往是 **一个 mini-batch（小批量）** 的数据（例如 32、64 或 128 个样本）。框架会计算这一个小批量的平均梯度来更新参数。这种做法的数学公式为：

$\theta = \theta - \eta \cdot \frac{1}{B} \sum_{i=1}^{B} abla J_i(\theta)$

（其中 $B$ 是 batch size，即小批量大小）

这本质上是 **小批量梯度下降（MBGD）**，但由于工程习惯，大家仍统称它为 SGD。如果想严格使用“1个样本”的纯 SGD，只需将 `batch_size` 设为 1 即可。

---

如果你还想了解 **带动量的 SGD（SGD with Momentum）** 公式，或者 **Adam** 优化器的更新方式，我可以继续为你推导。😊

SGD(随机梯度下降) 相比 GD(梯度下降) 每次参数更新时，不再计算整个数据集的梯度，而是仅随机抽取一个样本 \( (x^{(i)}, y^{(i)}) \) 来计算梯度。\
Adam 优化器是对梯度对参数的优化器，相比 SDG(随机梯度下降, $\theta_{j}^{(t+1)} = \theta_{j}^{(t)} - \eta \cdot \frac{\partial J}{\partial \theta_j}(\theta^{(t)})$ ) ,它引入了衰减系数，让历史梯度的影响逐步降低，只关注最近一段时间的梯度大小(梯度是反向传播[反向求导]传播后计算出的最能放大损失函数的)，从而让学习率始终保持活性

In [ ]:
class MyAdam:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        """
        Adam 优化器
        
        Args:
            params: 可迭代的模型参数（要求每个参数有 .data 和 .grad 属性）
            lr: 学习率
            betas: (beta1, beta2)，一阶矩和二阶矩的衰减率
            eps: 数值稳定项，防止分母为零
        """
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.t = 0  # 时间步
        
        # 为每个参数初始化一阶矩和二阶矩估计
        self.m = [None] * len(self.params)  # 一阶矩（均值）
        self.v = [None] * len(self.params)  # 二阶矩（方差）
        
        for i, p in enumerate(self.params):
            self.m[i] = p.data.new_zeros(p.data.shape)  # 创建与参数形状相同的零张量
            self.v[i] = p.data.new_zeros(p.data.shape)
    
    def zero_grad(self):
        """将所有参数的梯度清零"""
        for p in self.params:
            if p.grad is not None:
                p.grad.data.zero_()
    
    def step(self):
        """
        执行一步参数更新
        使用 Adam 算法更新所有参数
        """
        self.t += 1  # 增加时间步
        
        # 计算偏差校正系数（提前计算，避免在循环中重复计算）
        beta1_pow = self.beta1 ** self.t
        beta2_pow = self.beta2 ** self.t
        
        for i, p in enumerate(self.params):
            # 确保参数有梯度
            if p.grad is None:
                continue
            
            grad = p.grad.data
            
            # 更新一阶矩估计（动量）
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            
            # 更新二阶矩估计（RMS）
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * (grad * grad)
            
            # 偏差校正
            m_hat = self.m[i] / (1 - beta1_pow)
            v_hat = self.v[i] / (1 - beta2_pow)
            
            # 更新参数
            p.data -= self.lr * m_hat / (v_hat.sqrt() + self.eps)

In [ ]:
# 🧪 调试
torch.manual_seed(0)
w = torch.randn(4, 3, requires_grad=True)
opt = MyAdam([w], lr=0.01)
for i in range(5):
    loss = (w ** 2).sum()
    loss.backward()
    opt.step()
    opt.zero_grad()
    print(f'步骤 {i}: 损失={loss.item():.4f}')

In [ ]:
# ✅ 提交
from torch_judge import check
check('adam')